# SQL Analysis for Movie Investors  

In [2]:
import sqlite3
import pandas as pd

# Load cleaned dataset
csv_path = "tmdb_movies_cleaned (1).csv"
df = pd.read_csv(csv_path)

# Create in-memory SQLite database
conn = sqlite3.connect(":memory:")

# Write main table
df.to_sql("tmdb_movies_cleaned", conn, if_exists="replace", index=False)

# Helper function to run SQL and display results
def run_query(query, limit=None):
    result = pd.read_sql_query(query, conn)
    if limit is not None:
        return result.head(limit)
    return result

# Create helper views for cleaner joins
conn.executescript("""
DROP VIEW IF EXISTS vw_movie_core;
CREATE VIEW vw_movie_core AS
SELECT
    movie_id,
    title,
    release_date,
    release_year,
    release_month,
    original_language,
    language_group,
    primary_genre,
    runtime_group,
    high_rating,
    adult,
    status
FROM tmdb_movies_cleaned;

DROP VIEW IF EXISTS vw_movie_finance;
CREATE VIEW vw_movie_finance AS
SELECT
    movie_id,
    budget,
    revenue,
    profit,
    roi,
    blockbuster,
    budget_per_minute
FROM tmdb_movies_cleaned;

DROP VIEW IF EXISTS vw_movie_engagement;
CREATE VIEW vw_movie_engagement AS
SELECT
    movie_id,
    popularity,
    vote_average,
    vote_count,
    genre_count,
    production_company_count,
    production_country_count,
    spoken_language_count
FROM tmdb_movies_cleaned;
""")

print("Setup complete.")
print(f"Rows loaded: {len(df):,}")
print(f"Columns loaded: {len(df.columns)}")

Setup complete.
Rows loaded: 1,980
Columns loaded: 33


# Section 1. Is the dataset usable for investor analysis?

This section checks whether the key financial fields needed for investor analysis are sufficiently available and then defines the investable sample used throughout the rest of the notebook.

## Query 1. Financial data availability by release status

**Type:** GROUP BY / DATA QUALITY CHECK

**What this does:** Counts movies by `status` and measures how often `budget` and `revenue` are missing in each status group.

**How:** Uses `GROUP BY status`, `COUNT(*)`, and `SUM(CASE WHEN ... IS NULL THEN 1 ELSE 0 END)` to count missing financial values, then computes `AVG(vote_average)` for context.

**Why:** Before making investor recommendations, we need to know whether commercial performance can be measured reliably and whether missingness is concentrated in titles that are not yet fully released.

**General takeaway:** Missing financial data is substantial even among released films: **1,529 of 1,955 released titles** are missing budget and **1,541** are missing revenue. Missingness is even more concentrated in unreleased titles, with **all 14 `Post Production` films** missing both fields. That supports narrowing the investor analysis to a much smaller investable sample with realized financial outcomes.

In [3]:
query = """
SELECT
    status,
    COUNT(*) AS movie_count,
    SUM(CASE WHEN budget IS NULL THEN 1 ELSE 0 END) AS missing_budget,
    SUM(CASE WHEN revenue IS NULL THEN 1 ELSE 0 END) AS missing_revenue,
    ROUND(AVG(vote_average), 2) AS avg_vote_average
FROM tmdb_movies_cleaned
GROUP BY status
ORDER BY movie_count DESC;
"""

run_query(query)

,status,movie_count,missing_budget,missing_revenue,avg_vote_average
0,Released,1955,1529,1541,5.06
1,Post Production,14,14,14,0.00
2,In Production,9,7,9,0.00
3,Planned,1,1,1,0.00
4,Canceled,1,1,1,0.00


## Query 2. Define the investable sample

**Type:** FILTERING / SAMPLE DEFINITION

**What this does:** Quantifies how many movies remain after restricting the dataset to released films with non-null `budget` and `revenue` and positive budget.

**How:** Uses conditional aggregation with `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` to count total movies, released movies, and investable movies, then calculates the investable share as a percentage of the full sample.

**Why:** Investors care about realized commercial outcomes. This query defines the exact sample used in the rest of the notebook so later profitability comparisons are based on comparable, financially interpretable observations.

**General takeaway:** Only **353 of 1,980 movies**—about **17.83%** of the full dataset—meet the investable-screen criteria. So the later investor conclusions are based on a much smaller but much more reliable realized-performance sample.

In [4]:
query = """
SELECT
    COUNT(*) AS total_movies,
    SUM(CASE WHEN status = 'Released' THEN 1 ELSE 0 END) AS released_movies,
    SUM(CASE WHEN status = 'Released'
              AND budget IS NOT NULL
              AND revenue IS NOT NULL
              AND budget > 0
             THEN 1 ELSE 0 END) AS investable_movies,
    ROUND(
        100.0 * SUM(CASE WHEN status = 'Released'
                           AND budget IS NOT NULL
                           AND revenue IS NOT NULL
                           AND budget > 0
                          THEN 1 ELSE 0 END) / COUNT(*), 2
    ) AS investable_share_pct
FROM tmdb_movies_cleaned;
"""

run_query(query)

,total_movies,released_movies,investable_movies,investable_share_pct
0,1980,1955,353,17.83


# Section 2. What does commercial success look like overall?

This section establishes a baseline for the investable sample before comparing different movie characteristics.

## Query 3. Baseline commercial performance of the investable sample

**Type:** AGGREGATE SUMMARY

**What this does:** Summarizes average budget, revenue, profit, ROI, rating, vote count, and blockbuster share for the investable sample.

**How:** Filters to released movies with non-null `budget` and `revenue` and `budget > 0`, then uses aggregate functions such as `AVG()` and `COUNT()` to produce baseline commercial-performance benchmarks.

**Why:** Investors need a market baseline before asking which movie characteristics outperform the typical project. This query establishes the average commercial profile that later segment-level queries can be compared against.

**General takeaway:** The investable sample averages about 71.M budget, 334.6M revenue, and 263.5M profit, with a 59% blockbuster rate and an average rating of **7.11**. The mean ROI is extremely high at **1,508.87x**, which signals that a few very low-budget breakouts materially skew the average upward; investors should therefore interpret ROI patterns alongside profit, revenue, and blockbuster rate rather than in isolation.

In [5]:
query = """
SELECT
    COUNT(*) AS investable_movies,
    ROUND(AVG(budget), 2) AS avg_budget,
    ROUND(AVG(revenue), 2) AS avg_revenue,
    ROUND(AVG(profit), 2) AS avg_profit,
    ROUND(AVG(roi), 2) AS avg_roi,
    ROUND(AVG(blockbuster), 2) AS blockbuster_rate,
    ROUND(AVG(vote_average), 2) AS avg_rating,
    ROUND(AVG(vote_count), 2) AS avg_vote_count
FROM tmdb_movies_cleaned
WHERE status = 'Released'
  AND budget IS NOT NULL
  AND revenue IS NOT NULL
  AND budget > 0;
"""

run_query(query)

,investable_movies,avg_budget,avg_revenue,avg_profit,avg_roi,blockbuster_rate,avg_rating,avg_vote_count
0,353,71118616.81,3.346352e+08,2.635166e+08,1508.87,0.59,7.11,7369.91


# Section 3. Which movie characteristics are associated with stronger returns?

This section compares commercial performance across genre, language, runtime, and audience response.

## Query 4. Which genres look most attractive for investors?

**Type:** GROUP BY #1

**What this does:** Compares average budget, revenue, profit, ROI, blockbuster rate, and rating across `primary_genre`.

**How:** Filters to the investable sample, groups by `primary_genre`, computes averages for financial and audience metrics, and uses `HAVING COUNT(*) >= 3` to remove tiny categories that could distort investor interpretation.

**Why:** Genre is one of the most direct content-allocation decisions an investor can make. This query tests whether some genres consistently look more commercially attractive than others.

**General takeaway:** **Animation** stands out most strongly on return, with about **15,610.77x average ROI** and a **79% blockbuster rate**, but that average is helped by several extremely high-multiple anime titles. Among bigger, more scalable categories, **Adventure** looks especially attractive with roughly 627.9M average revenue, 505.8M average profit, and a 79% blockbuster rate, while **Family** also looks strong on both ROI (**10.34x**) and hit rate (**74%**). By contrast, **History** and **Science Fiction** look weaker on ROI in this sample despite sizable budgets.

In [6]:
query = """
SELECT
    primary_genre,
    COUNT(*) AS movies,
    ROUND(AVG(budget), 2) AS avg_budget,
    ROUND(AVG(revenue), 2) AS avg_revenue,
    ROUND(AVG(profit), 2) AS avg_profit,
    ROUND(AVG(roi), 2) AS avg_roi,
    ROUND(AVG(blockbuster), 2) AS blockbuster_rate,
    ROUND(AVG(vote_average), 2) AS avg_rating
FROM tmdb_movies_cleaned
WHERE status = 'Released'
  AND budget IS NOT NULL
  AND revenue IS NOT NULL
  AND budget > 0
GROUP BY primary_genre
HAVING COUNT(*) >= 3
ORDER BY avg_roi DESC, blockbuster_rate DESC;
"""

run_query(query)

,primary_genre,movies,avg_budget,avg_revenue,avg_profit,avg_roi,blockbuster_rate,avg_rating
0,Animation,34,9.130000e+07,5.390748e+08,4.477748e+08,15610.77,0.79,7.68
1,Western,5,3.316000e+07,1.491169e+08,1.159569e+08,23.98,0.40,7.75
2,Family,19,7.296933e+07,4.654412e+08,3.924719e+08,10.34,0.74,7.06
3,Adventure,33,1.221702e+08,6.279245e+08,5.057543e+08,7.39,0.79,7.33
4,Horror,34,3.233853e+07,1.365197e+08,1.041812e+08,6.97,0.47,6.61
5,Thriller,8,6.417500e+07,2.201231e+08,1.559481e+08,5.80,0.75,7.19
6,Crime,16,4.199375e+07,2.068534e+08,1.648596e+08,5.77,0.44,7.48
7,Action,63,1.077095e+08,4.833418e+08,3.756322e+08,4.99,0.67,7.08
8,Drama,58,3.557612e+07,1.697043e+08,1.341282e+08,4.99,0.45,7.33
9,Comedy,40,3.586192e+07,1.716198e+08,1.357579e+08,4.70,0.40,6.54


## Query 5. Does language group matter for commercial success?

**Type:** GROUP BY #2

**What this does:** Compares commercial performance by `language_group`.

**How:** Filters to the investable sample, groups by `language_group`, computes average budget, revenue, profit, ROI, blockbuster rate, and rating, and applies `HAVING COUNT(*) >= 3` to keep only groups with a minimally usable sample size.

**Why:** For investors, language can proxy for market reach and international monetization potential. This query tests whether the commercial profile differs across language segments.

**General takeaway:** **Japanese-language** films post the highest average ROI (**106,122.87x**) and an **80% blockbuster rate**, but that result comes from only **5 films** and is heavily influenced by a few low-budget anime breakouts. For scalable investing, **English-language** films are far more important: they account for **304 investable titles**, average about 370.2M revenue and 290.1M profit, and convert to blockbusters **64%** of the time. **French-language** films look weakest commercially in this sample, with only **1.22x ROI** and a **10%** blockbuster rate.

In [7]:
query = """
SELECT
    language_group,
    COUNT(*) AS movies,
    ROUND(AVG(budget), 2) AS avg_budget,
    ROUND(AVG(revenue), 2) AS avg_revenue,
    ROUND(AVG(profit), 2) AS avg_profit,
    ROUND(AVG(roi), 2) AS avg_roi,
    ROUND(AVG(blockbuster), 2) AS blockbuster_rate,
    ROUND(AVG(vote_average), 2) AS avg_rating
FROM tmdb_movies_cleaned
WHERE status = 'Released'
  AND budget IS NOT NULL
  AND revenue IS NOT NULL
  AND budget > 0
GROUP BY language_group
HAVING COUNT(*) >= 3
ORDER BY avg_roi DESC, blockbuster_rate DESC;
"""

run_query(query)

,language_group,movies,avg_budget,avg_revenue,avg_profit,avg_roi,blockbuster_rate,avg_rating
0,Japanese,5,10120016.80,3.264675e+08,3.163475e+08,106122.87,0.80,7.75
1,Korean,3,10154333.33,1.112488e+08,1.010945e+08,10.46,0.33,7.75
2,Other,28,15516857.14,1.179598e+08,1.024429e+08,9.09,0.18,6.85
3,Spanish,3,2633333.33,1.710929e+07,1.447596e+07,6.88,0.00,7.02
4,English,304,80102587.66,3.701750e+08,2.900724e+08,5.59,0.64,7.12
5,French,10,23025000.00,2.727442e+07,4.249419e+06,1.22,0.10,7.11


## Query 6. Does runtime group affect returns?

**Type:** GROUP BY #3

**What this does:** Compares budget, revenue, profit, ROI, blockbuster rate, and rating across `runtime_group`.

**How:** Filters to the investable sample, groups by `runtime_group`, and computes average commercial and audience metrics for each runtime bucket.

**Why:** Runtime can shape production scope, release strategy, and audience accessibility. Investors may want to know whether longer or shorter films tend to produce better financial outcomes.

**General takeaway:** On raw mean ROI, Short films look extreme at **20,411.42x**, but that figure is clearly driven by outlier low-budget hits. For a more scalable investor lens, **Long** films look strongest: they average about 476.3M revenue, 379.6M profit, a **74% blockbuster rate**, and the highest average rating (**7.46**). **Medium** films trail long films on both hit rate and profit.

In [8]:
query = """
SELECT
    runtime_group,
    COUNT(*) AS movies,
    ROUND(AVG(budget), 2) AS avg_budget,
    ROUND(AVG(revenue), 2) AS avg_revenue,
    ROUND(AVG(profit), 2) AS avg_profit,
    ROUND(AVG(roi), 2) AS avg_roi,
    ROUND(AVG(blockbuster), 2) AS blockbuster_rate,
    ROUND(AVG(vote_average), 2) AS avg_rating
FROM tmdb_movies_cleaned
WHERE status = 'Released'
  AND budget IS NOT NULL
  AND revenue IS NOT NULL
  AND budget > 0
GROUP BY runtime_group
ORDER BY avg_roi DESC;
"""

run_query(query)

,runtime_group,movies,avg_budget,avg_revenue,avg_profit,avg_roi,blockbuster_rate,avg_rating
0,Short,26,29049130.15,1.608640e+08,1.318149e+08,20411.42,0.54,6.54
1,Long,159,96639490.57,4.762723e+08,3.796328e+08,6.52,0.74,7.46
2,Medium,168,53475686.61,2.274791e+08,1.740034e+08,5.35,0.45,6.86


## Query 7. Are stronger audience signals linked to stronger financial outcomes?

**Type:** GROUP BY #4 / AUDIENCE SIGNAL TEST

**What this does:** Compares movies above and below the `high_rating` threshold on rating, vote count, revenue, profit, ROI, and blockbuster rate.

**How:** Filters to the investable sample, groups by the binary flag `high_rating`, and computes average commercial and audience metrics for each group.

**Why:** Investors often treat audience response as an early signal of market quality. This query tests whether stronger audience ratings are associated with better realized financial outcomes.

**General takeaway:** Audience quality looks meaningfully connected to commercial performance. The `high_rating = 1` group averages about 450.6M revenue, 365.3M profit, and a **71% blockbuster rate**, versus 168.3M revenue, 117.5M profit, and **41%** for lower-rated titles. The lower-rated group’s mean ROI is inflated by an outlier, so the stronger investor signal here is the much higher profit scale and hit rate for highly rated films.

In [9]:
query = """
SELECT
    high_rating,
    COUNT(*) AS movies,
    ROUND(AVG(vote_average), 2) AS avg_rating,
    ROUND(AVG(vote_count), 2) AS avg_vote_count,
    ROUND(AVG(revenue), 2) AS avg_revenue,
    ROUND(AVG(profit), 2) AS avg_profit,
    ROUND(AVG(roi), 2) AS avg_roi,
    ROUND(AVG(blockbuster), 2) AS blockbuster_rate
FROM tmdb_movies_cleaned
WHERE status = 'Released'
  AND budget IS NOT NULL
  AND revenue IS NOT NULL
  AND budget > 0
GROUP BY high_rating
ORDER BY high_rating;
"""

run_query(query)

,high_rating,movies,avg_rating,avg_vote_count,avg_revenue,avg_profit,avg_roi,blockbuster_rate
0,0,145,6.25,2517.92,1.682877e+08,1.175070e+08,3661.88,0.41
1,1,208,7.71,10752.31,4.505986e+08,3.653022e+08,7.98,0.71


# Section 4. What combinations of characteristics look most promising?

This section uses joins and subqueries to move beyond one-variable comparisons and identify combinations of characteristics that repeatedly appear in stronger performers.

## Query 8. Join finance and engagement: do movies with more audience traction outperform financially?

**Type:** JOIN #1 + WINDOW FUNCTION #1

**What this does:** Joins the finance and engagement views, sorts movies into vote-count quartiles, and compares financial outcomes across those audience-traction buckets.

**How:** Uses `JOIN` to combine `vw_movie_finance`, `vw_movie_engagement`, and `vw_movie_core` on `movie_id`, applies `NTILE(4) OVER (ORDER BY vote_count)` to create vote-count quartiles, and then aggregates revenue, profit, ROI, rating, and blockbuster rate by quartile.

**Why:** This query shows whether audience traction is just correlated with popularity metrics or whether it is also linked to stronger realized commercial performance—something directly relevant for investors screening opportunities.

**General takeaway:** The relationship is strongly positive. Movies in the **top vote-count quartile** average about 703.1M revenue, 593.4M profit, **9.48x ROI**, and a **93% blockbuster rate**, versus only 56.7M revenue and an **11%** blockbuster rate in the **bottom quartile**. That makes audience traction one of the clearest investor-relevant signals in the dataset.

In [10]:
query = """
WITH joined_data AS (
    SELECT
        f.movie_id,
        f.revenue,
        f.profit,
        f.roi,
        f.blockbuster,
        e.vote_average,
        e.vote_count,
        NTILE(4) OVER (ORDER BY e.vote_count) AS vote_count_quartile
    FROM vw_movie_finance f
    JOIN vw_movie_engagement e
      ON f.movie_id = e.movie_id
    JOIN vw_movie_core c
      ON f.movie_id = c.movie_id
    WHERE c.status = 'Released'
      AND f.budget IS NOT NULL
      AND f.revenue IS NOT NULL
      AND f.budget > 0
)
SELECT
    vote_count_quartile,
    COUNT(*) AS movies,
    ROUND(AVG(vote_average), 2) AS avg_rating,
    ROUND(AVG(revenue), 2) AS avg_revenue,
    ROUND(AVG(profit), 2) AS avg_profit,
    ROUND(AVG(roi), 2) AS avg_roi,
    ROUND(AVG(blockbuster), 2) AS blockbuster_rate
FROM joined_data
GROUP BY vote_count_quartile
ORDER BY vote_count_quartile;
"""

run_query(query)

,vote_count_quartile,movies,avg_rating,avg_revenue,avg_profit,avg_roi,blockbuster_rate
0,1,89,6.50,5.669677e+07,3.431302e+07,5963.12,0.11
1,2,88,6.91,2.010684e+08,1.365348e+08,4.63,0.43
2,3,88,7.14,3.808325e+08,2.924007e+08,7.64,0.88
3,4,88,7.90,7.031016e+08,5.934225e+08,9.48,0.93


## Query 9. Join core, finance, and engagement: which genre-language combinations look strongest?

**Type:** JOIN #2

**What this does:** Combines genre, language, finance, and engagement variables to compare average ROI, profit, blockbuster rate, and rating across `primary_genre × language_group` combinations.

**How:** Uses `JOIN` across `vw_movie_core`, `vw_movie_finance`, and `vw_movie_engagement`, groups by `primary_genre` and `language_group`, and applies `HAVING COUNT(*) >= 2` to keep combinations with at least a minimal repeated pattern.

**Why:** Investors rarely evaluate one characteristic in isolation. This query looks for multi-factor combinations that appear commercially attractive enough to matter in portfolio construction.

**General takeaway:** The most explosive ROI combination is **Japanese Animation** (**5 films**, **106,122.87x ROI**, **80%** blockbuster rate), but among more scalable segments the strongest repeated combinations are **English Family** (**17 films**, **11.32x ROI**, **76%** blockbuster rate), **English Adventure** (**32 films**, **7.55x ROI**, **81%** blockbuster rate), and **English Thriller** (**86%** blockbuster rate). **English Action** also remains attractive on scale, averaging about 418.9M profit across **56 films**.

In [11]:
query = """
SELECT
    c.primary_genre,
    c.language_group,
    COUNT(*) AS movies,
    ROUND(AVG(f.roi), 2) AS avg_roi,
    ROUND(AVG(f.profit), 2) AS avg_profit,
    ROUND(AVG(f.blockbuster), 2) AS blockbuster_rate,
    ROUND(AVG(e.vote_average), 2) AS avg_rating
FROM vw_movie_core c
JOIN vw_movie_finance f
  ON c.movie_id = f.movie_id
JOIN vw_movie_engagement e
  ON c.movie_id = e.movie_id
WHERE c.status = 'Released'
  AND f.budget IS NOT NULL
  AND f.revenue IS NOT NULL
  AND f.budget > 0
GROUP BY c.primary_genre, c.language_group
HAVING COUNT(*) >= 2
ORDER BY avg_roi DESC, blockbuster_rate DESC;
"""

run_query(query)

,primary_genre,language_group,movies,avg_roi,avg_profit,blockbuster_rate,avg_rating
0,Animation,Japanese,5,106122.87,3.163475e+08,0.80,7.75
1,Western,Other,2,52.46,2.600000e+07,0.00,8.15
2,Animation,Other,2,14.22,1.085956e+09,0.50,7.41
3,Comedy,Korean,2,12.86,1.355917e+08,0.50,8.03
4,Horror,Other,3,11.84,5.144806e+06,0.00,6.57
5,Family,English,17,11.32,4.337660e+08,0.76,7.09
6,Adventure,English,32,7.55,5.209218e+08,0.81,7.35
7,Horror,English,30,6.69,1.181711e+08,0.53,6.60
8,Thriller,English,7,6.45,1.780692e+08,0.86,7.27
9,Crime,English,14,6.18,1.856383e+08,0.50,7.42


## Query 10. Subquery: what traits appear most often among above-average ROI movies?

**Type:** SUBQUERY #1

**What this does:** Keeps only movies whose ROI is above the investable-sample average, then groups those above-average performers by genre, runtime, and language.

**How:** Uses a scalar subquery in the `WHERE` clause—`roi > (SELECT AVG(roi) ...)`—to isolate above-average ROI titles, then aggregates the surviving records by `primary_genre`, `runtime_group`, and `language_group`.

**Why:** Investors care more about the traits that recur among above-market opportunities than about raw averages across the full dataset. This query focuses the lens on the winners.

**General takeaway:** This query reveals an important data feature: the overall mean ROI is so heavily skewed by extreme low-budget winners that only **one film** clears the sample-average ROI threshold. That surviving case is a **Japanese short-form Animation** title. For investors, the lesson is not that this exact niche is broadly repeatable, but that ROI is highly outlier-sensitive in this dataset and should be paired with scale metrics like profit and blockbuster rate.

In [12]:
query = """
SELECT
    primary_genre,
    runtime_group,
    language_group,
    COUNT(*) AS above_avg_roi_movies,
    ROUND(AVG(roi), 2) AS avg_roi,
    ROUND(AVG(blockbuster), 2) AS blockbuster_rate
FROM tmdb_movies_cleaned
WHERE status = 'Released'
  AND budget IS NOT NULL
  AND revenue IS NOT NULL
  AND budget > 0
  AND roi > (
      SELECT AVG(roi)
      FROM tmdb_movies_cleaned
      WHERE status = 'Released'
        AND budget IS NOT NULL
        AND revenue IS NOT NULL
        AND budget > 0
  )
GROUP BY primary_genre, runtime_group, language_group
ORDER BY above_avg_roi_movies DESC, avg_roi DESC;
"""

run_query(query)

,primary_genre,runtime_group,language_group,above_avg_roi_movies,avg_roi,blockbuster_rate
0,Animation,Short,Japanese,1,530466.61,0.0


# Section 5. What patterns should investors remember?

This final section uses ranking logic to identify benchmark winners and repeatable group-level patterns.

## Query 11. Window function: which movies are the top ROI performers within each genre?

**Type:** JOIN #3 + WINDOW FUNCTION #2

**What this does:** Ranks movies by ROI within each genre so that each title can be compared against the strongest performer in its own segment.

**How:** Uses `JOIN` between `vw_movie_core` and `vw_movie_finance`, then applies `RANK() OVER (PARTITION BY primary_genre ORDER BY roi DESC)` to create a genre-specific ROI ranking while preserving title-level detail.

**Why:** Window functions are valuable because they keep row-level information while adding relative standing inside a category. For investors, this helps identify benchmark titles for each genre rather than only looking at market-wide winners.

**General takeaway:** The genre leaders include several very high-multiple benchmark titles: **Prisoner of War** leads Action at **47.35x ROI**, **Star Wars** leads Adventure at **70.49x**, **JUJUTSU KAISEN: Execution** leads Animation at **530,466.61x**, and **The Jungle Book** leads Family at **94.50x**. These benchmarks help investors understand what “best-in-class” looks like inside each segment, while also highlighting how extreme some low-budget winners can be.

In [13]:
query = """
WITH ranked_movies AS (
    SELECT
        c.primary_genre,
        c.title,
        c.language_group,
        c.runtime_group,
        f.budget,
        f.revenue,
        f.profit,
        f.roi,
        RANK() OVER (
            PARTITION BY c.primary_genre
            ORDER BY f.roi DESC
        ) AS genre_roi_rank
    FROM vw_movie_core c
    JOIN vw_movie_finance f
      ON c.movie_id = f.movie_id
    WHERE c.status = 'Released'
      AND f.budget IS NOT NULL
      AND f.revenue IS NOT NULL
      AND f.budget > 0
)
SELECT
    primary_genre,
    title,
    language_group,
    runtime_group,
    ROUND(budget, 2) AS budget,
    ROUND(revenue, 2) AS revenue,
    ROUND(profit, 2) AS profit,
    ROUND(roi, 2) AS roi
FROM ranked_movies
WHERE genre_roi_rank = 1
ORDER BY roi DESC;
"""

run_query(query)

,primary_genre,title,language_group,runtime_group,budget,revenue,profit,roi
0,Animation,JUJUTSU KAISEN: Execution,Japanese,Short,84.0,4.455920e+07,4.455911e+07,530466.61
1,Family,The Jungle Book,English,Short,4000000.0,3.780000e+08,3.740000e+08,94.50
2,Western,A Fistful of Dollars,Other,Medium,200000.0,1.450000e+07,1.430000e+07,72.50
3,Adventure,Star Wars,English,Long,11000000.0,7.753980e+08,7.643980e+08,70.49
4,Action,Prisoner of War,English,Medium,2000000.0,9.470000e+07,9.270000e+07,47.35
5,Drama,The Godfather,English,Long,6000000.0,2.450664e+08,2.390664e+08,40.84
6,Horror,The Exorcist,English,Long,12000000.0,4.413061e+08,4.293061e+08,36.78
7,Thriller,Pulp Fiction,English,Long,8000000.0,2.139288e+08,2.059288e+08,26.74
8,Comedy,Parasite,Korean,Long,11363000.0,2.575918e+08,2.462288e+08,22.67
9,Crime,Joker,English,Long,55000000.0,1.078959e+09,1.023959e+09,19.62


## Query 12. Subquery + window function: which scalable segment patterns look strongest?

**Type:** SUBQUERY #2 + WINDOW FUNCTION #3

**What this does:** First computes group-level performance for `primary_genre × language_group`, filters to combinations with at least two movies, and then ranks the surviving groups by ROI and blockbuster rate.

**How:** Uses a `group_stats` CTE to aggregate each genre-language segment, then a second step with `DENSE_RANK() OVER (ORDER BY avg_roi DESC)` and `DENSE_RANK() OVER (ORDER BY blockbuster_rate DESC, avg_profit DESC)` to rank segments on return and commercial hit rate while keeping only repeatable groups.

**Why:** Investors should care more about scalable patterns than about one-off winners. This query surfaces segments that appear both commercially attractive and repeated often enough to be usable for portfolio thinking.

**General takeaway:** On pure ROI, **Japanese Animation** ranks first, but the more scalable English-language patterns are especially useful for investors: **English Family** ranks highest on ROI among larger English segments, **English Adventure** combines strong ROI with high profits and an **81%** blockbuster rate, and **English Animation** stands out most on blockbuster consistency with a **92%** hit rate. These segments look more actionable than isolated outliers.

In [14]:
query = """
WITH group_stats AS (
    SELECT
        c.primary_genre,
        c.language_group,
        COUNT(*) AS movies,
        AVG(f.roi) AS avg_roi,
        AVG(f.blockbuster) AS blockbuster_rate,
        AVG(f.profit) AS avg_profit,
        AVG(e.vote_average) AS avg_rating
    FROM vw_movie_core c
    JOIN vw_movie_finance f
      ON c.movie_id = f.movie_id
    JOIN vw_movie_engagement e
      ON c.movie_id = e.movie_id
    WHERE c.status = 'Released'
      AND f.budget IS NOT NULL
      AND f.revenue IS NOT NULL
      AND f.budget > 0
    GROUP BY c.primary_genre, c.language_group
),
ranked_groups AS (
    SELECT
        primary_genre,
        language_group,
        movies,
        ROUND(avg_roi, 2) AS avg_roi,
        ROUND(blockbuster_rate, 2) AS blockbuster_rate,
        ROUND(avg_profit, 2) AS avg_profit,
        ROUND(avg_rating, 2) AS avg_rating,
        DENSE_RANK() OVER (ORDER BY avg_roi DESC) AS roi_rank,
        DENSE_RANK() OVER (ORDER BY blockbuster_rate DESC, avg_roi DESC) AS blockbuster_rank
    FROM group_stats
    WHERE movies >= 2
)
SELECT
    primary_genre,
    language_group,
    movies,
    avg_roi,
    blockbuster_rate,
    avg_profit,
    avg_rating,
    roi_rank,
    blockbuster_rank
FROM ranked_groups
ORDER BY roi_rank, blockbuster_rank;
"""

run_query(query)

,primary_genre,language_group,movies,avg_roi,blockbuster_rate,avg_profit,avg_rating,roi_rank,blockbuster_rank
0,Animation,Japanese,5,106122.87,0.80,3.163475e+08,7.75,1,4
1,Western,Other,2,52.46,0.00,2.600000e+07,8.15,2,22
2,Animation,Other,2,14.22,0.50,1.085956e+09,7.41,3,14
3,Comedy,Korean,2,12.86,0.50,1.355917e+08,8.03,4,15
4,Horror,Other,3,11.84,0.00,5.144806e+06,6.57,5,23
5,Family,English,17,11.32,0.76,4.337660e+08,7.09,6,5
6,Adventure,English,32,7.55,0.81,5.209218e+08,7.35,7,3
7,Horror,English,30,6.69,0.53,1.181711e+08,6.60,8,12
8,Thriller,English,7,6.45,0.86,1.780692e+08,7.27,9,2
9,Crime,English,14,6.18,0.50,1.856383e+08,7.42,10,16


# Investor takeaways

### What the data suggests

Based on the investable sample used in this notebook, the strongest investor-facing patterns are more specific than the generic statement that “multiple factors matter.”

**1. The most scalable opportunities in this dataset are concentrated in English-language Family, Adventure, Animation, Action, and Thriller films.**  
The single most explosive ROI pockets come from **Japanese Animation** and a few other small non-English groups, but those categories are based on very small samples and are heavily affected by extreme low-budget winners. For a more repeatable investor lens, the stronger patterns come from larger **English-language** segments. In particular, **English Family** averaged about **11.32x ROI** across **17 films**, **English Adventure** averaged **7.55x ROI** across **32 films** with an **81% blockbuster rate**, **English Animation** posted a remarkable **92% blockbuster rate** across **24 films**, and **English Action** generated about 418.9M average profit across **56 films**.

**2. Audience traction is one of the clearest commercial signals in the data.**  
The quartile analysis shows a steep climb in outcomes as vote count rises. Movies in the **top vote-count quartile** averaged roughly 703.1M revenue, 593.4M profit, and a **93% blockbuster rate**, compared with only 56.7M revenue and an **11%** blockbuster rate in the bottom quartile. Similarly, highly rated movies averaged 450.6M revenue and a **71% blockbuster rate**, versus 168.3M and **41%** for lower-rated titles. For investors, that means audience traction is not just a vanity metric; it is strongly linked to realized monetization.

**3. Long films look more investable than short films once you focus on scalable outcomes rather than extreme ROI outliers.**  
Short films show the highest mean ROI, but that result is dominated by a single ultra-low-budget breakout and is not representative of the broader sample. By contrast, **Long** films averaged about 476.3M revenue, 379.6M profit, and a **74% blockbuster rate**, materially ahead of **Medium** films on both scale and consistency. That makes long-format projects look stronger for investors seeking repeatable upside rather than lottery-ticket outcomes.

**4. Some categories look materially weaker and deserve more caution.**  
**French-language** films look weak on average, with only **1.22x ROI** and a **10%** blockbuster rate. Genre-wise, **History** and **Science Fiction** also look less efficient on ROI than Family, Adventure, or Animation despite often requiring meaningful budgets; for example, **Science Fiction** averaged only **2.44x ROI** even with very high average budgets. These categories may still work selectively, but they appear harder to back on average in this sample.

**5. The dataset also warns investors not to over-rely on mean ROI.**  
A few ultra-low-budget winners push the average ROI to extreme levels: the investable sample’s mean ROI is **1,508.87x**, and only **one film** actually exceeds that average. That is a strong signal that ROI alone can be misleading here. Investors should place more weight on **profit, revenue, blockbuster rate, and the consistency of repeated segment-level patterns** when screening opportunities.